# Analyse exploratoire du développement africain

Ce notebook explore les données nettoyées de la Banque mondiale pour 54 pays africains sur la période 2000–2024. Il couvre les statistiques descriptives, les distributions, les corrélations, les tendances temporelles, les comparaisons régionales, les outliers et la génération automatique de `INSIGHTS.md`. 

**Source :** Banque mondiale / World Development Indicators.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "african_development_clean.csv"
df = pd.read_csv(DATA_PATH, keep_default_na=False)

print("Dimensions:", df.shape)
print("Période:", int(df["year"].min()), "–", int(df["year"].max()))
print("Nombre de pays:", df["country"].nunique())
df.head()

Dimensions: (1350, 31)
Période: 2000 – 2024
Nombre de pays: 54


,country,country_code,region,income_level,year,electricity_access,gdp,gdp_per_capita,inflation_rate,internet_users,...,electricity_access_was_imputed,gdp_was_imputed,gdp_per_capita_was_imputed,inflation_rate_was_imputed,internet_users_was_imputed,life_expectancy_was_imputed,population_was_imputed,unemployment_rate_was_imputed,gdp_growth_yoy,population_growth_yoy
0,Algeria,DZ,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,2000,98.6,5.479040e+10,1772.928691,0.339163,0.491706,...,False,False,False,False,False,False,False,False,,
1,Algeria,DZ,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,2001,98.6,5.941340e+10,1896.300209,4.225988,0.646114,...,False,False,False,False,False,False,False,False,8.437614023512122,1.3827643009248058
2,Algeria,DZ,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,2002,98.6,6.151610e+10,1937.464114,1.418302,1.591641,...,False,False,False,False,False,False,False,False,3.5391047303199996,1.3392839047032352
3,Algeria,DZ,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,2003,98.6,7.348226e+10,2283.772993,4.268954,2.195360,...,False,False,False,False,False,False,False,False,19.45207859761114,1.3384939325217746
4,Algeria,DZ,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,2004,98.6,9.191368e+10,2816.993850,3.961800,4.634475,...,False,False,False,False,False,False,False,False,25.0828101136855,1.4062361988745709


## Inspection et qualité des données

Les données nettoyées doivent contenir 54 pays, 25 années et aucun doublon pays/année. Les valeurs imputées sont suivies par les colonnes `*_was_imputed`; les potentiels outliers sont suivis par `*_is_outlier`.

In [3]:
print("Types de données:")
print(df.dtypes)

print("\nValeurs manquantes:")
print(df.isna().sum().sort_values(ascending=False).head(15))

print("\nDoublons pays/année:", df.duplicated(["country", "year"]).sum())
print("Doublons complets:", df.duplicated().sum())

Types de données:
country                               str
country_code                          str
region                                str
income_level                          str
year                                int64
electricity_access                float64
gdp                               float64
gdp_per_capita                    float64
inflation_rate                    float64
internet_users                    float64
life_expectancy                   float64
population                        float64
unemployment_rate                 float64
electricity_access_is_outlier        bool
gdp_is_outlier                       bool
gdp_per_capita_is_outlier            bool
inflation_rate_is_outlier            bool
internet_users_is_outlier            bool
life_expectancy_is_outlier           bool
population_is_outlier                bool
unemployment_rate_is_outlier         bool
electricity_access_was_imputed       bool
gdp_was_imputed                      bool
gdp_per_capita_w

## Statistiques descriptives et distributions

In [4]:
numeric_columns = [
    "electricity_access", "gdp", "gdp_per_capita", "inflation_rate",
    "internet_users", "life_expectancy", "population", "unemployment_rate",
    "gdp_growth_yoy", "population_growth_yoy",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

statistics = df[numeric_columns].describe().T
statistics["median"] = df[numeric_columns].median()
statistics = statistics[["count", "mean", "median", "std", "min", "25%", "50%", "75%", "max"]]
statistics.round(3)

,count,mean,median,std,min,25%,50%,75%,max
electricity_access,1350.0,4.569900e+01,4.295000e+01,2.959500e+01,8.000000e-01,1.870000e+01,4.295000e+01,6.600000e+01,1.000000e+02
gdp,1350.0,3.783080e+10,1.146559e+10,8.202956e+10,7.595113e+07,3.226008e+09,1.146559e+10,2.919394e+10,6.682200e+11
gdp_per_capita,1350.0,2.278814e+03,1.081963e+03,2.944299e+03,1.095940e+02,5.758200e+02,1.081963e+03,2.724603e+03,1.914151e+04
inflation_rate,1350.0,1.011600e+01,5.418000e+00,3.143500e+01,-1.686000e+01,2.581000e+00,5.418000e+00,9.492000e+00,5.572020e+02
internet_users,1350.0,1.651600e+01,7.000000e+00,2.050300e+01,6.000000e-03,1.543000e+00,7.000000e+00,2.463500e+01,9.120000e+01
life_expectancy,1350.0,6.045300e+01,6.041400e+01,7.664000e+00,1.466500e+01,5.535300e+01,6.041400e+01,6.522600e+01,7.723700e+01
population,1350.0,2.117413e+07,1.133039e+07,3.136295e+07,8.113100e+04,2.487068e+06,1.133039e+07,2.539802e+07,2.326795e+08
unemployment_rate,1350.0,9.106000e+00,5.783000e+00,7.337000e+00,3.160000e-01,3.280000e+00,5.783000e+00,1.400200e+01,3.647200e+01
gdp_growth_yoy,1266.0,8.279000e+00,7.473000e+00,1.666800e+01,-6.103200e+01,4.920000e-01,7.473000e+00,1.440700e+01,1.644380e+02
population_growth_yoy,1296.0,2.385000e+00,2.495000e+00,1.205000e+00,-5.272000e+00,1.785000e+00,2.495000e+00,3.008000e+00,2.077400e+01


In [5]:
fig = px.histogram(df, x="gdp", nbins=40, marginal="box", title="Distribution du PIB")
fig.show()

fig = px.histogram(df, x="gdp_per_capita", nbins=40, marginal="box", title="Distribution du PIB par habitant")
fig.show()

## Corrélations

La matrice utilise la corrélation de Pearson. Une valeur proche de 1 indique une relation positive, proche de -1 une relation négative et proche de 0 une relation linéaire faible.

In [6]:
correlations = df[numeric_columns].corr(numeric_only=True)

fig = px.imshow(
    correlations,
    text_auto=".2f",
    aspect="auto",
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    title="Matrice de corrélation",
)
fig.show()

correlations.round(3)

,electricity_access,gdp,gdp_per_capita,inflation_rate,internet_users,life_expectancy,population,unemployment_rate,gdp_growth_yoy,population_growth_yoy
electricity_access,1.000,0.352,0.629,-0.091,0.662,0.729,0.079,0.368,-0.088,-0.389
gdp,0.352,1.000,0.181,0.010,0.308,0.139,0.756,0.133,-0.027,-0.093
gdp_per_capita,0.629,0.181,1.000,-0.070,0.518,0.466,-0.114,0.331,-0.017,-0.205
inflation_rate,-0.091,0.010,-0.070,1.000,-0.055,-0.103,0.079,0.003,-0.036,-0.037
internet_users,0.662,0.308,0.518,-0.055,1.000,0.587,0.058,0.248,-0.164,-0.306
life_expectancy,0.729,0.139,0.466,-0.103,0.587,1.000,-0.012,0.124,-0.122,-0.196
population,0.079,0.756,-0.114,0.079,0.058,-0.012,1.000,-0.176,-0.012,0.081
unemployment_rate,0.368,0.133,0.331,0.003,0.248,0.124,-0.176,1.000,-0.043,-0.286
gdp_growth_yoy,-0.088,-0.027,-0.017,-0.036,-0.164,-0.122,-0.012,-0.043,1.000,0.152
population_growth_yoy,-0.389,-0.093,-0.205,-0.037,-0.306,-0.196,0.081,-0.286,0.152,1.000


## Top 10, Flop 10 et évolution temporelle

In [7]:
latest_year = int(df["year"].max())
latest = df[df["year"] == latest_year]

top10 = latest.nlargest(10, "gdp")[["country", "gdp", "gdp_per_capita", "population", "life_expectancy"]]
flop10 = latest.nsmallest(10, "gdp")[["country", "gdp", "gdp_per_capita", "population", "life_expectancy"]]

print(f"Top 10 du PIB — {latest_year}")
display(top10)
print(f"Flop 10 du PIB — {latest_year}")
display(flop10)

temporal = df.groupby("year").agg(
    gdp_mean=("gdp", "mean"),
    gdp_median=("gdp", "median"),
    gdp_total=("gdp", "sum"),
    population_total=("population", "sum"),
    electricity_access_mean=("electricity_access", "mean"),
    internet_users_mean=("internet_users", "mean"),
    life_expectancy_mean=("life_expectancy", "mean"),
    unemployment_rate_mean=("unemployment_rate", "mean"),
).reset_index()

temporal_fig = px.line(
    temporal,
    x="year",
    y=["electricity_access_mean", "internet_users_mean", "life_expectancy_mean"],
    title="Évolution des indicateurs sociaux moyens",
)
temporal_fig.show()
temporal.round(3).tail()

Top 10 du PIB — 2024


,country,gdp,gdp_per_capita,population,life_expectancy
1149,South Africa,4.011450e+11,6267.186814,64007187.0,66.312
399,"Egypt, Arab Rep.",3.890599e+11,3338.473710,116538258.0,71.806
24,Algeria,2.693223e+11,5752.990767,46814308.0,76.475
974,Nigeria,2.522619e+11,1084.160418,232679478.0,54.635
874,Morocco,1.606110e+11,4153.194336,38081173.0,75.493
499,Ethiopia,1.497403e+11,1133.882797,132059767.0,67.603
649,Kenya,1.203975e+11,2133.461934,56432944.0,63.834
49,Angola,1.030805e+11,2720.819007,37885849.0,64.805
349,Cote d'Ivoire,8.711318e+10,2727.893522,31934230.0,62.110
574,Ghana,8.328859e+10,2419.251867,34427414.0,65.695


Flop 10 du PIB — 2024


,country,gdp,gdp_per_capita,population,life_expectancy
1024,Sao Tome and Principe,8.249926e+08,3502.617681,235536.0,69.905000
449,Eritrea,1.160108e+09,437.419176,3535603.0,68.893000
274,Comoros,1.610083e+09,1857.870607,866628.0,67.023000
624,Guinea-Bissau,2.197777e+09,998.376093,2201352.0,64.254000
1074,Seychelles,2.228609e+09,18364.525968,121354.0,76.295122
674,Lesotho,2.391283e+09,1023.042277,2337423.0,57.798000
549,"Gambia, The",2.404889e+09,871.340288,2759988.0,66.057000
174,Cabo Verde,2.713722e+09,5170.205319,524877.0,76.224000
224,Central African Republic,2.751494e+09,516.161000,5330690.0,57.671000
149,Burundi,3.037580e+09,216.231929,14047786.0,63.818000


,year,gdp_mean,gdp_median,gdp_total,population_total,electricity_access_mean,internet_users_mean,life_expectancy_mean,unemployment_rate_mean
20,2020,4.949498e+10,1.425483e+10,2.672729e+12,1.378200e+09,55.389,34.818,63.904,9.617
21,2021,5.529614e+10,1.652000e+10,2.985991e+12,1.411047e+09,56.819,37.330,63.164,9.745
22,2022,5.967808e+10,1.762696e+10,3.222616e+12,1.444092e+09,57.826,39.664,64.186,9.112
23,2023,5.571076e+10,1.887066e+10,3.008381e+12,1.477881e+09,59.870,39.917,65.441,9.020
24,2024,5.402079e+10,1.981825e+10,2.917123e+12,1.512197e+09,61.885,42.490,65.688,9.028


## Comparaison régionale et détection IQR

Les données utilisent les deux grandes régions de la Banque mondiale. Le notebook les regroupent aussi en cinq sous-régions africaines pour une comparaison plus lisible.

In [8]:
regional = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "comparaison_regions.csv")
regional

fig = px.bar(
    regional,
    x="african_subregion",
    y="gdp_per_capita_mean",
    color="african_subregion",
    title="PIB moyen par habitant selon la sous-région",
)
fig.show()

outlier_columns = [
    "electricity_access", "gdp", "gdp_per_capita", "inflation_rate",
    "internet_users", "life_expectancy", "population", "unemployment_rate",
]
outlier_counts = {}

for column in outlier_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_counts[column] = int(((df[column] < lower) | (df[column] > upper)).sum())

pd.Series(outlier_counts).sort_values(ascending=False)

gdp                   176
gdp_per_capita        138
inflation_rate        111
population            104
internet_users         83
life_expectancy         9
unemployment_rate       9
electricity_access      0
dtype: int64

## Synthèse des cinq conclusions

In [9]:
first_year = temporal.iloc[0]
last_year = temporal.iloc[-1]

internet_growth = last_year["internet_users_mean"] - first_year["internet_users_mean"]
electricity_growth = last_year["electricity_access_mean"] - first_year["electricity_access_mean"]
life_growth = last_year["life_expectancy_mean"] - first_year["life_expectancy_mean"]

north = regional.loc[regional["african_subregion"] == "Afrique du Nord"].iloc[0]
west = regional.loc[regional["african_subregion"] == "Afrique de l'Ouest"].iloc[0]

first_gdp = float(top10.iloc[0]["gdp"])
lowest_gdp = float(flop10.iloc[0]["gdp"])
ratio_gdp = first_gdp / lowest_gdp

conclusions = [
    f"L'utilisation moyenne d'Internet a progressé de {first_year['internet_users_mean']:.2f} % en 2000 à {last_year['internet_users_mean']:.2f} % en 2024, soit une hausse de {internet_growth:.2f} points.",
    f"L'accès moyen à l'électricité est passé de {first_year['electricity_access_mean']:.2f} % à {last_year['electricity_access_mean']:.2f} %, une augmentation de {electricity_growth:.2f} points qui accompagne une hausse de l'espérance de vie de {life_growth:.2f} ans.",
    f"La corrélation entre l'accès à l'électricité et l'espérance de vie est positive et forte (r = {correlations.loc['electricity_access', 'life_expectancy']:.3f}); cette relation indique une tendance commune, sans prouver une causalité.",
    f"Le PIB moyen par habitant en Afrique du Nord ({north['gdp_per_capita_mean']:,.0f} USD) est environ {(north['gdp_per_capita_mean'] / west['gdp_per_capita_mean']):.1f} fois celui observé en Afrique de l'Ouest ({west['gdp_per_capita_mean']:,.0f} USD), un écart régional important.",
    f"En 2024, le PIB de {top10.iloc[0]['country']} ({first_gdp/1e9:.1f} milliards USD) était environ {ratio_gdp:.0f} fois celui de {flop10.iloc[0]['country']} ({lowest_gdp/1e9:.3f} milliard USD), illustrant une forte concentration économique africaine.",
]

insights_path = PROJECT_ROOT / "INSIGHTS.md"
sections = ["# Conclusions de l'analyse exploratoire", ""]
sections.append("Les conclusions suivantes sont directement étayées par les fichiers CSV produits après l'analyse exploratoire.")
sections.append("")
for index, conclusion in enumerate(conclusions, start=1):
    sections.append(f"{index}. {conclusion}")

insights_path.write_text("\n".join(sections), encoding="utf-8")
print(insights_path)

for index, conclusion in enumerate(conclusions, start=1):
    print(f"{index}. {conclusion}")

/home/emmanuella/s2-dashboard-streamlit/INSIGHTS.md
1. L'utilisation moyenne d'Internet a progressé de 0.89 % en 2000 à 42.49 % en 2024, soit une hausse de 41.60 points.
2. L'accès moyen à l'électricité est passé de 33.36 % à 61.89 %, une augmentation de 28.52 points qui accompagne une hausse de l'espérance de vie de 10.64 ans.
3. La corrélation entre l'accès à l'électricité et l'espérance de vie est positive et forte (r = 0.729); cette relation indique une tendance commune, sans prouver une causalité.
4. Le PIB moyen par habitant en Afrique du Nord (3,786 USD) est environ 3.3 fois celui observé en Afrique de l'Ouest (1,141 USD), un écart régional important.
5. En 2024, le PIB de South Africa (401.1 milliards USD) était environ 486 fois celui de Sao Tome and Principe (0.825 milliard USD), illustrant une forte concentration économique africaine.
